In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BorsaBot — 04 | Model Eğitimi
# RegimeDetector (HMM)  →  PrimaryModel (XGBoost)  →  MetaModel (LightGBM)
# ═══════════════════════════════════════════════════════════════════
# labeled, ohlcv, CFG mevcut olmalı.

In [ ]:
!pip install -q xgboost lightgbm hmmlearn scikit-learn imbalanced-learn

import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
# ─────────────────────────────────────────────────────────────────
from hmmlearn.hmm import GaussianHMM

def train_regime_detector(close_series: pd.Series, n_states: int = 3):
    """
    Gaussian HMM ile 3 rejim öğren:
      low_vol  → range piyasa
      trending → trend piyasa
      high_vol → kriz / yüksek vol
    """
    ret = close_series.pct_change().dropna().values
    obs = np.column_stack([ret, np.abs(ret), np.sign(ret)])

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=200,
        random_state=42,
    )
    model.fit(obs)

    # Durumları volatiliteye göre sırala (low → high)
    states = model.predict(obs)
    mean_vol = {s: float(np.mean(np.abs(ret[states == s])))
                for s in range(n_states)}
    sorted_s = sorted(mean_vol, key=mean_vol.get)
    mapping  = {sorted_s[0]: "low_vol",
                sorted_s[1]: "trending",
                sorted_s[2]: "high_vol"}

    return model, mapping, states

# Tüm sembollerin close'unu birleştir
all_close = pd.concat([ohlcv[s]["close"] for s in CFG["symbols"]]).sort_index()
regime_model, regime_mapping, _ = train_regime_detector(all_close)

print("✓ Rejim Dedektörü eğitildi")
print(f"  Durum eşlemesi: {regime_mapping}")

# Kaydet
with open(f"{CFG['model_dir']}/regime.pkl", "wb") as f:
    pickle.dump({"model": regime_model, "mapping": regime_mapping}, f)
print("  → regime.pkl kaydedildi")

In [ ]:
# ─────────────────────────────────────────────────────────────────
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
LABEL_RMAP = {0: -1, 1: 0, 2: 1}
FEAT_COLS  = [c for c in list(labeled[CFG["symbols"][0]].columns)
              if c not in ["label", "t1", "trgt"]]

def train_primary_model(sym: str):
    df   = labeled[sym].copy()
    X    = df[FEAT_COLS].values
    y    = df["label"].map(LABEL_MAP).values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.15, shuffle=False   # zaman serisi: shuffle=False
    )

    # SMOTE ile sınıf dengeleme
    try:
        sm = SMOTE(random_state=42, k_neighbors=3)
        X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
    except Exception as e:
        print(f"  SMOTE atlandı: {e}")

    clf = XGBClassifier(
        n_estimators     = 400,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 5,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        num_class        = 3,
        objective        = "multi:softprob",
        eval_metric      = "mlogloss",
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )
    clf.fit(X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            verbose=False)

    y_pred = clf.predict(X_te)
    print(f"\n[{sym}] Primary Model Raporu:")
    print(classification_report(
        y_te, y_pred,
        target_names=["SELL(-1)", "FLAT(0)", "BUY(+1)"]
    ))

    # Feature importance grafiği
    imp = pd.Series(clf.feature_importances_, index=FEAT_COLS).sort_values()
    plt.figure(figsize=(8, 5))
    imp.tail(15).plot.barh()
    plt.title(f"{sym} — Primary Model Feature Importance")
    plt.tight_layout()
    plt.savefig(f"{CFG['out_dir']}/{sym}_primary_importance.png", dpi=80)
    plt.close()

    return clf

primary_models = {}
for sym in CFG["symbols"]:
    print(f"\nEğitiliyor: {sym} PrimaryModel ...")
    clf = train_primary_model(sym)
    primary_models[sym] = clf

    with open(f"{CFG['model_dir']}/{sym}_primary.pkl", "wb") as f:
        pickle.dump({"model": clf,
                     "feat_cols": FEAT_COLS,
                     "label_map": LABEL_MAP,
                     "label_rmap": LABEL_RMAP}, f)
    print(f"  → {sym}_primary.pkl kaydedildi")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Meta label: primary model o sampleda doğru mu tahmin etti?  {0,1}

def make_meta_labels(sym: str, primary_clf) -> pd.DataFrame:
    df   = labeled[sym].copy()
    X    = df[FEAT_COLS].values
    y    = df["label"].map(LABEL_MAP).values
    pred = primary_clf.predict(X)

    meta_y = (pred == y).astype(int)  # 1=doğru, 0=yanlış
    # Sadece primary'nin FLAT olmadığı yerler
    non_flat = pred != 1
    df_meta  = df[non_flat].copy()
    df_meta["meta_label"]     = meta_y[non_flat]
    df_meta["primary_pred"]   = np.vectorize(LABEL_RMAP.get)(pred[non_flat])
    proba    = primary_clf.predict_proba(X[non_flat])
    df_meta["primary_conf"]   = proba.max(axis=1)

    return df_meta

meta_dfs = {}
for sym in CFG["symbols"]:
    meta_dfs[sym] = make_meta_labels(sym, primary_models[sym])
    pos_rate = meta_dfs[sym]["meta_label"].mean()
    print(f"[{sym}] Meta dataset: {len(meta_dfs[sym]):,} sample  |  "
          f"Doğru oranı: {pos_rate:.1%}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
from lightgbm import LGBMClassifier

def train_meta_model(sym: str):
    df = meta_dfs[sym].copy()

    meta_feat_cols = FEAT_COLS + ["primary_pred", "primary_conf"]
    X  = df[meta_feat_cols].values
    y  = df["meta_label"].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.15, shuffle=False
    )

    # Scale negatif weight ile dengeleme
    pos_w = (y_tr == 0).sum() / (y_tr == 1).sum()

    clf = LGBMClassifier(
        n_estimators     = 400,
        num_leaves       = 31,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        min_child_samples= 20,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        scale_pos_weight = pos_w,
        random_state     = 42,
        n_jobs           = -1,
        verbose          = -1,
    )
    clf.fit(X_tr, y_tr,
            eval_set=[(X_te, y_te)])

    y_pred = clf.predict(X_te)
    print(f"\n[{sym}] Meta Model Raporu:")
    print(classification_report(y_te, y_pred,
                                 target_names=["WRONG", "CORRECT"]))
    return clf, meta_feat_cols

meta_models = {}
for sym in CFG["symbols"]:
    print(f"\nEğitiliyor: {sym} MetaModel ...")
    clf, mfeat = train_meta_model(sym)
    meta_models[sym] = clf

    with open(f"{CFG['model_dir']}/{sym}_meta.pkl", "wb") as f:
        pickle.dump({"model": clf, "feat_cols": mfeat}, f)
    print(f"  → {sym}_meta.pkl kaydedildi")

print("\n✓ Tüm modeller eğitildi ve kaydedildi!")
print(f"  Konum: {CFG['model_dir']}/")